# 📋 NLP Final Ödev-2 — 01: Veri Ön İşleme

**Proje:** Endüstriyel Arıza Kodları Eşleştirme

**Grup:** Rodolfo Mba NDONG MEBAHA

Bu notebook ham arıza kodu verilerini yükleyip Türkçe NLP ön işleme adımlarını uygular ve `lemmatized.csv` ile `stemmed.csv` çıktı dosyalarını üretir.

## 1. Kütüphane Kurulumu ve İçe Aktarma

In [ ]:
# Gerekli kütüphaneleri kur (ilk çalıştırmada)
# !pip install gensim zeyrek snowballstemmer nltk pandas

import pandas as pd
import re
import nltk
import zeyrek
import snowballstemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('Tum kutuphaneler yuklendi.')

## 2. Ham Veriyi Yükle

In [ ]:
df_raw = pd.read_csv("data/fault_codes_raw.csv")
print(f"Toplam kayit: {len(df_raw)}")
print(f"Sutunlar    : {list(df_raw.columns)}")
df_raw.head(5)

## 3. Türkçe Stop-Words Listesi

In [ ]:
tr_stopwords = set(stopwords.words("turkish"))
extra = {"ve","ile","icin","olan","veya","ancak","ise","bu","bir","da","de","ki","cok","daha","en","gibi","kadar","var","yok","her","bazi","ayni","diger","tum","butun"}
tr_stopwords.update(extra)
print(f"Stop-word sayisi: {len(tr_stopwords)}")

## 4. Ön İşleme Fonksiyonları

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\u00c0-\u017f\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_tr(text):
    tokens = word_tokenize(text, language="turkish")
    return [t for t in tokens if t.isalpha() and t not in tr_stopwords and len(t) > 2]

analyzer = zeyrek.MorphAnalyzer()

def lemmatize_tr(tokens):
    lemmas = []
    for token in tokens:
        result = analyzer.lemmatize(token)
        if result and result[0][1]:
            lemmas.append(result[0][1][0].lower())
        else:
            lemmas.append(token)
    return lemmas

stemmer_tr = snowballstemmer.stemmer("turkish")

def stem_tr(tokens):
    return [stemmer_tr.stemWord(t) for t in tokens]

print("On isleme fonksiyonlari hazir.")
ornek = "Motor asiri isinma hatasi tespit edildi."
cleaned = clean_text(ornek)
tokens  = tokenize_tr(cleaned)
lemmas  = lemmatize_tr(tokens)
stems   = stem_tr(tokens)
print(f"Ham     : {ornek}")
print(f"Temizle : {cleaned}")
print(f"Tokenler: {tokens}")
print(f"Lemma   : {lemmas}")
print(f"Stem    : {stems}")

## 5. Tüm Veri Setine Ön İşleme Uygula

In [ ]:
lemmatized_docs = []
stemmed_docs    = []

for idx, row in df_raw.iterrows():
    doc_id  = row["document_id"]
    content = row["content"]
    cleaned = clean_text(content)
    tokens  = tokenize_tr(cleaned)
    lemmas  = lemmatize_tr(tokens)
    stems   = stem_tr(tokens)
    lemmatized_docs.append({"document_id": doc_id, "content": " ".join(lemmas)})
    stemmed_docs.append   ({"document_id": doc_id, "content": " ".join(stems)})
    if (idx + 1) % 20 == 0:
        print(f"  {idx+1}/{len(df_raw)} kayit islendi...")

print(f"Toplam {len(df_raw)} kayit islendi.")

## 6. CSV Olarak Kaydet

In [ ]:
df_lemmatized = pd.DataFrame(lemmatized_docs)
df_stemmed    = pd.DataFrame(stemmed_docs)

df_lemmatized.to_csv("data/lemmatized.csv", index=False, encoding="utf-8")
df_stemmed.to_csv   ("data/stemmed.csv",    index=False, encoding="utf-8")

print("Dosyalar kaydedildi:")
print("  data/lemmatized.csv")
print("  data/stemmed.csv")

## 7. Boyut ve Yapı Karşılaştırması

In [ ]:
print(f"{"Veri Seti":<20} {"Kayit":<10} {"Ort. Kelime/Belge":<22} {"Toplam Kelime"}")
print("-" * 65)
for name, df in [("Ham Veri", df_raw), ("Lemmatized", df_lemmatized), ("Stemmed", df_stemmed)]:
    wc = df["content"].apply(lambda x: len(str(x).split()))
    print(f"{name:<20} {len(df):<10} {wc.mean():<22.1f} {wc.sum()}")

## 8. Örnek Çıktı Karşılaştırması

In [ ]:
print("HAM vs LEMMATIZED vs STEMMED (ilk 3 belge)")
print("=" * 65)
for i in range(3):
    print(f"\n{df_raw.iloc[i]["document_id"]}")
    print(f"  HAM  : {df_raw.iloc[i]["content"][:80]}...")
    print(f"  LEMMA: {df_lemmatized.iloc[i]["content"][:80]}...")
    print(f"  STEM : {df_stemmed.iloc[i]["content"][:80]}...")